# Module 8 — The Wealth-Signal Demo with Bounded Agent

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

This is the capstone. Every prior module built a piece of the architecture. Module 8
stitches them together into the workshop's reference application: a wealth-signal
detection workflow that identifies a Consumer-side customer with in-bank wealth
indicators, scores the lead, presents it to a bounded agent for routing, and surfaces
the routed lead to a human Wealth advisor for approval.

By the end of this module you can:

- Explain the end-to-end wealth-signal workflow to a business or technical audience
- Explain how the bounded agent selects routes from an enumerated set (not by LLM reasoning)
- Show the full audit trail from signal detection through advisor approval in SPARQL
- Explain why every component in the workflow is classified as deterministic,
  probabilistic-explainable, or probabilistic-opaque

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **Bounded agent** | A Step Functions state machine (optionally orchestrated by Bedrock AgentCore) that selects from a finite, declaratively-defined set of routing actions. The LLM inside the runtime interprets context but does not enlarge the route set. |
| **AWS Step Functions** | A serverless workflow orchestration service. You define a state machine (states + transitions), and Step Functions executes it. In ATLAS, the state machine IS the bounded agent. |
| **Human-in-the-loop (HITL)** | A workflow pattern where an automated process pauses for human review before a consequential action. In ATLAS, the advisor reviews and approves/declines each routed lead. |
| **Task token** | A Step Functions mechanism for pausing a workflow until an external system (the reviewer UI) signals completion. The workflow waits; the human decides; the workflow resumes. |
| **XGBoost** | Extreme Gradient Boosting — a gradient-boosted tree algorithm used in the SageMaker scoring path. Produces deterministic-given-version outputs that pair with SHAP for explainability. |
| **SHAP (SHapley Additive exPlanations)** | A feature-attribution method that explains which input features drove a model's output for a specific record. Required for every Score in ATLAS. |
| **Amazon Bedrock AgentCore** | AWS service for building agents with explicit tool declarations. In ATLAS, demonstrates the same bounded-agent pattern as Step Functions with a different orchestration surface. |
| **EventBridge** | Amazon EventBridge — a serverless event bus. In ATLAS, fires when a wealth-eligibility event lands in the LGD and when a routing decision is approved. |
| **AppSync** | AWS AppSync — a managed GraphQL service. Backs the reviewer UI that lists pending wealth leads with their evidence chains. |
| **Alex Morgan** | The synthetic Wealth advisor persona who reviews leads in the demo. The only named individual in the workshop. |

## The end-to-end workflow

```
1. EventBridge rule fires (wealth-eligibility event in LGD)
       ↓
2. Step Functions state machine (bounded agent) orchestrates:
   a. Enrich event with household context from SLGD
   b. Call SageMaker XGBoost endpoint for score + SHAP
   c. Invoke Bedrock to draft a contact note (for human review)
   d. Select routing target from enumerated set
   e. Pause for human approval (task token)
       ↓
3. Advisor (Alex Morgan) reviews in Streamlit UI:
   - Sees SHAP-attributed score
   - Sees in-bank evidence chain
   - Sees Bedrock-drafted contact note
   - Approves or declines with comment
       ↓
4. On approval:
   - State machine writes routing decision to SLGD with PROV-O
   - Fires outbound EventBridge event for downstream CRM
   - Closes the workflow
```

## The single rule that governs agents in ATLAS

> **LLMs do not make routing decisions; agents execute routing decisions, where the
> decision logic is deterministic and the LLM's role inside the agent is interface,
> not reasoning.**

The LLM inside the agent runtime:
- Reads tool outputs (SPARQL results, XGBoost scores)
- Helps the agent choose which tool to call next from a declared set
- Drafts narratives for the human reviewer

The LLM does NOT:
- Select routes by free reasoning
- Invent tools
- Make compliance decisions
- Enlarge the route set beyond what the state machine declares

## Prerequisites

- All prior modules complete (Modules 1–7)
- Neptune clusters running with ontology and promoted data
- Bedrock access for narrative drafting

## Deliverables

- A simulated end-to-end workflow execution
- The full audit trail queryable in SPARQL
- A walkthrough suitable for presenting to a business or technical audience

## Architecture class for this module

**MIXED.** The workflow contains all three component classes:
- DETERMINISTIC: SHACL validation, route selection from enumerated set, SPARQL queries
- PROBABILISTIC-EXPLAINABLE: XGBoost scoring with SHAP attributions
- PROBABILISTIC-OPAQUE: Bedrock narrative drafting (interface role only, not a decision input)

In [ ]:
# Workshop dependency setup — installs into THIS kernel's Python interpreter.
# Uses absolute path + cwd='/tmp' to avoid pip's os.getcwd() failure in SageMaker.
import sys, subprocess, os

req = os.path.join(os.getcwd(), 'shared', 'requirements.txt')
if not os.path.exists(req):
    # Fallback: install the key packages directly
    pkgs = ['rdflib==7.0.0', 'pyshacl==0.25.0', 'requests==2.31.0',
            'numpy==1.26.4', 'pandas==2.2.2', 'pyarrow==14.0.2',
            'faker==24.3.0', 'xgboost==2.1.4', 'scikit-learn==1.5.2']
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs,
        cwd='/tmp'
    )
else:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', req],
        cwd='/tmp'
    )
print('Dependencies ready.')


## How This Connects to Competency Questions

This is the payoff. The Competency Questions (CQs) you wrote in Module 1 were
acceptance tests for an empty ontology. In Module 8, those same questions are
answered by a running system with real (synthetic) data flowing through it.

The audit trail query in this module is essentially
**CQ6** ("What is the full audit trail from signal detection to advisor approval?")
running against the SLGD (Semantic Layer Graph Database) with promoted, scored,
routed, and reviewed data. One SPARQL query. Full audit trail. Every component
classified.

This completes the Competency Question lifecycle:

| Module | CQ Role | What Happens |
|--------|---------|-------------|
| 1 | **Validation** | CQs prove the ontology has the right structure |
| 2 | **Stability** | CQs remain valid after FIBO (Financial Industry Business Ontology) alignment |
| 3 | **Physical** | CQs run against live Neptune infrastructure |
| 4 | **Data** | Each data pattern feeds specific CQs |
| 5 | **Derivation** | CQs answered by computed (not loaded) data |
| 6 | **Enforcement** | SHACL (Shapes Constraint Language) enforces what CQs imply |
| 7 | **Grounding** | CQs become few-shot examples and accuracy benchmarks |
| 8 | **Proof of value** | CQs answered end-to-end in a live demo ← you are here |

In [ ]:
import sys
sys.path.insert(0, '../notebooks/shared')

import json
from datetime import datetime
from pathlib import Path
import random
import atlas_synthetic
import atlas_sparql

print('Module 8 — The Wealth-Signal Demo with Bounded Agent')
print(f'Synthetic data seed: {atlas_synthetic.ATLAS_SEED}')
print(f'Demo persona: Alex Morgan (Wealth Advisor)')

ATLAS_NS = 'https://github.com/your-org/atlas/ontology#'
INST_NS = 'https://github.com/your-org/atlas/instance#'
XSD_NS = 'http://www.w3.org/2001/XMLSchema#'
RDF_TYPE = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'

## Simulating the End-to-End Workflow

In production, this workflow runs as a Step Functions state machine triggered by
EventBridge. For the workshop, we simulate each step in sequence to show what
happens at each stage and what data flows between components.

### Step 1: A wealth-eligibility event arrives

A customer's deposit-balance trajectory crosses the large-deposit threshold (a value your risk team calibrates for your book).
The transaction monitor fires an event to the LGD.

In [ ]:
# Step 1: A wealth-eligibility event arrives
rng = random.Random(atlas_synthetic.ATLAS_SEED)

# Pick a customer with a large-deposit signal
customers = atlas_synthetic.generate_customers(n=200)
accounts = atlas_synthetic.generate_accounts(customers)
transactions = atlas_synthetic.generate_transactions(accounts, lookback_days=90)

# Find a signal transaction
signal_txn = next(t for t in transactions if t.get('signal_tag') == 'large-deposit-pattern')
target_customer = next(c for c in customers if c['customer_id'] == signal_txn['customer_id'])

print('Step 1: Wealth-Eligibility Event Detected')
print('=' * 60)
print(f'  Customer:    {target_customer["first_name"]} {target_customer["last_name"]}')
print(f'  Customer ID: {target_customer["customer_id"][:12]}...')
print(f'  Segment:     {target_customer["segment"]}')
print(f'  Household:   {target_customer["household_id"][:12]}...')
print(f'  Signal type: large-deposit-pattern')
print(f'  Amount:      ${signal_txn["amount_usd"]:,.2f}')
print(f'  Date:        {signal_txn["transaction_date"]}')
print()
print('  Event written to LGD as atlas:BehavioralEvent')
print('  EventBridge rule fires -> Step Functions state machine starts')

### Step 2: The bounded agent enriches and scores

The state machine:
1. Queries the SLGD for household context (other members, combined balance)
2. Scores the customer with an XGBoost wealth-conversion model
3. Attaches SHAP feature attributions to the score

In production, step 2 calls a SageMaker endpoint hosting a model trained and
validated offline. In this workshop, the next cell trains a small XGBoost model
inline on the 200 synthetic customers — a minimum-viable model that demonstrates
the mechanism and produces the real SHAP attributions the SHACL `ComplianceInputShape`
requires. The model is trained with a fixed seed, so every run produces the same
score.

In [ ]:
# Step 2: Enrich and Score
#
# In production, the wealth-conversion model is trained offline and deployed to a
# SageMaker endpoint; the bounded agent calls that endpoint. Here we train a small
# XGBoost model INLINE on this workshop's 200 synthetic customers so the notebook
# is self-contained and fully reproducible. This is a minimum-viable model on a toy
# dataset to demonstrate the mechanism and the SHAP explainability that the SHACL
# ComplianceInputShape requires — NOT a validated production model.
import numpy as np
import xgboost as xgb
from datetime import date

print('Step 2: Bounded Agent Enriches and Scores')
print('=' * 60)

# --- Household enrichment (queried from the SLGD in production) ---
household_members = [c for c in customers if c['household_id'] == target_customer['household_id']]
household_balance = sum(
    a['balance_usd'] for a in accounts
    if a['customer_id'] in [m['customer_id'] for m in household_members]
)
print(f'  Household enrichment:')
print(f'    Members in household: {len(household_members)}')
print(f'    Combined balance:     ${household_balance:,.2f}')
print()

# --- Build a customer-level feature matrix and train the model ---
# Features are customer-level aggregates (not the single triggering transaction),
# so no single feature trivially encodes the label.
FEATURES = ['max_deposit_90d', 'household_balance', 'account_tenure_years',
            'segment_affluent', 'prior_surfacing_none']
DEPOSIT_TYPES = {'DEPOSIT', 'LARGE_DEPOSIT'}
TODAY = date.today()

# Precompute per-customer aggregates
_accts_by_cust = {}
for a in accounts:
    _accts_by_cust.setdefault(a['customer_id'], []).append(a)
_hh_balance = {}
for c in customers:
    _members = [m['customer_id'] for m in customers if m['household_id'] == c['household_id']]
    _hh_balance[c['customer_id']] = sum(
        a['balance_usd'] for a in accounts if a['customer_id'] in _members
    )
_max_dep = {}
for t in transactions:
    if t['transaction_type'] in DEPOSIT_TYPES:
        _cid = t['customer_id']
        _max_dep[_cid] = max(_max_dep.get(_cid, 0.0), t['amount_usd'])

def _tenure_years(cid):
    opens = [a['opened_date'] for a in _accts_by_cust.get(cid, [])]
    if not opens:
        return 0.0
    earliest = min(date.fromisoformat(o) for o in opens)
    return round((TODAY - earliest).days / 365.25, 2)

def _feature_row(c):
    cid = c['customer_id']
    return [
        _max_dep.get(cid, 0.0),
        _hh_balance.get(cid, 0.0),
        _tenure_years(cid),
        1.0 if c['segment'] == 'AFFLUENT' else 0.0,   # exact match; MASS_AFFLUENT is not AFFLUENT
        1.0,  # prior_surfacing_none: no prior-surfacing field in this dataset -> constant (near-zero SHAP, honestly)
    ]

# Label: customer exhibits ANY known wealth signal (transaction-tagged or household-level)
_signal_custs = set(t['customer_id'] for t in transactions if t.get('signal_tag'))
_hh_signal_hh = set(s['household_id'] for s in atlas_synthetic.generate_household_signals(customers, accounts))
def _has_signal(c):
    return c['customer_id'] in _signal_custs or c['household_id'] in _hh_signal_hh

X = np.array([_feature_row(c) for c in customers], dtype=float)
y = np.array([1 if _has_signal(c) else 0 for c in customers], dtype=int)

model = xgb.XGBClassifier(n_estimators=60, max_depth=3, learning_rate=0.2,
                          random_state=42, eval_metric='logloss', n_jobs=1)
model.fit(X, y)
print(f'  Model trained inline: {X.shape[0]} customers, {int(y.sum())} signal-positive')
print(f'    (XGBoost, fixed seed -> identical every run)')
print()

# --- Score the target customer ---
_x_target = np.array([_feature_row(target_customer)], dtype=float)
score_value = round(float(model.predict_proba(_x_target)[0, 1]), 3)

# --- SHAP attributions via XGBoost native pred_contribs (true Shapley values) ---
_booster = model.get_booster()
_dm = xgb.DMatrix(_x_target, feature_names=FEATURES)
_contribs = _booster.predict(_dm, pred_contribs=True)[0]   # len = n_features + 1 (last = base)
shap_features = {f: round(float(_contribs[i]), 4) for i, f in enumerate(FEATURES)}
shap_base = round(float(_contribs[-1]), 4)

print(f'  XGBoost Score:')
print(f'    Wealth-conversion probability: {score_value}')
print(f'    Model version: wealth-xgb-v1.0')
print(f'    Component class: PROBABILISTIC-EXPLAINABLE')
print()
print(f'  SHAP Feature Attributions (log-odds contribution to this prediction):')
for feature, contribution in sorted(shap_features.items(), key=lambda x: -abs(x[1])):
    sign = '+' if contribution >= 0 else '-'
    bar = '#' * int(min(abs(contribution), 2.0) * 20)
    print(f'    {feature:<22} {sign}{abs(contribution):.4f} {bar}')
print(f'    {"(base value)":<22}  {shap_base:+.4f}')
print()
print(f'  These are real Shapley values from the trained model (XGBoost pred_contribs).')
print(f'  In production they would be persisted alongside the Score for MRM review;')
print(f'  here they are shown to demonstrate the explainability the boundary requires.')


### Step 3: Route selection from the enumerated set

The bounded agent selects a route. The route set is **closed** — defined in the
state machine, enforced by the SHACL routing-policy shape from Module 6.

The three permissible routes:
- `ROUTE_ADVISOR_QUEUE` — send to advisor for review
- `ROUTE_SUPPRESSION_LIST` — do not contact (customer opted out or recently contacted)
- `ROUTE_ESCALATION` — escalate to senior advisor or compliance

The selection logic is deterministic: if score >= 0.7 and no prior suppression,
route to advisor queue. The LLM does not participate in this decision.

In [ ]:
# Step 3: Route Selection (DETERMINISTIC)
print('Step 3: Route Selection')
print('=' * 60)

# Deterministic routing logic (NOT LLM-driven)
if score_value >= 0.7:
    selected_route = 'ROUTE_ADVISOR_QUEUE'
    route_reason = f'Score {score_value} >= 0.7 threshold, no prior suppression'
elif score_value >= 0.5:
    selected_route = 'ROUTE_ESCALATION'
    route_reason = f'Score {score_value} in [0.5, 0.7) range, requires senior review'
else:
    selected_route = 'ROUTE_SUPPRESSION_LIST'
    route_reason = f'Score {score_value} < 0.5, below threshold'

print(f'  Selected route: {selected_route}')
print(f'  Reason:         {route_reason}')
print(f'  Component class: DETERMINISTIC (rule-based, not LLM)')
print()
print(f'  SHACL routing-policy shape check:')
print(f'    Route "{selected_route}" is in closed set: PASS')
print(f'    (Module 6 shape would reject any value not in the set)')

### Step 4: Human-in-the-loop review

The state machine pauses (via a task token) and presents the lead to the advisor.
Alex Morgan sees:
- The customer's name and segment
- The wealth-conversion score with SHAP attributions
- The in-bank evidence (the large deposit transaction)
- A Bedrock-drafted contact note (for review, not for direct use)

Alex approves or declines with a comment.

In [ ]:
# Step 4: Human Review (Alex Morgan)
print('Step 4: Human-in-the-Loop Review')
print('=' * 60)
print()
print('  Reviewer: Alex Morgan (Wealth Advisor)')
print('  Lead presented in reviewer UI:')
print(f'    Customer:  {target_customer["first_name"]} {target_customer["last_name"]}')
print(f'    Segment:   {target_customer["segment"]}')
print(f'    Score:     {score_value} (SHAP-explained)')
print(f'    Signal:    Large Deposit Pattern (${signal_txn["amount_usd"]:,.2f})')
print(f'    Route:     {selected_route}')
print()

# Simulate Bedrock drafting a contact note
contact_note = (
    f'{target_customer["first_name"]} {target_customer["last_name"]} recently deposited '
    f'${signal_txn["amount_usd"]:,.2f} into their checking account. Combined with a '
    f'household balance of ${household_balance:,.2f}, this suggests potential interest '
    f'in wealth management services. Recommend scheduling an introductory call.'
)
print(f'  Bedrock-drafted contact note (for review, not direct use):')
print(f'    "{contact_note}"')
print()
print(f'  Component class of contact note: PROBABILISTIC-OPAQUE')
print(f'  (Drafted by LLM, reviewed by human, not a compliance input)')
print()

# Simulate approval
review_outcome = 'APPROVED'
review_comment = 'Good candidate. Schedule intro call next week.'
review_timestamp = datetime.utcnow().isoformat() + 'Z'

print(f'  Alex Morgan\'s decision: {review_outcome}')
print(f'  Comment: "{review_comment}"')
print(f'  Timestamp: {review_timestamp}')
print()
print(f'  Task token released -> state machine resumes')

### Step 5: Write the audit trail to the SLGD

On approval, the state machine writes the complete routing decision to the SLGD
with full PROV-O provenance. This is the audit trail a regulator can query.

### Engagement vs Coverage: The Two Outputs of Module 8

When Alex Morgan approves a lead, the workflow produces **two** distinct outputs:

1. **HumanReview** (engagement event) — records *what happened*: who reviewed,
   when, what the outcome was. This is a PROV-O activity in the audit chain.
   It answers: "What actions were taken on this signal?"

2. **AdvisoryRelationship** (coverage assertion) — records *who is responsible*:
   which advisor now covers this customer, starting when, under what relationship
   type. This is a standing assignment that persists beyond the workflow.
   It answers: "Who is this customer's advisor right now?"

The distinction matters because:
- Engagement is an **event chain** — it has a start, a middle, and an end.
  Once the workflow closes, the HumanReview is historical.
- Coverage is a **standing assertion** — it remains active until explicitly ended.
  It is what the CRM queries when routing future communications.

Coverage is *derived from* engagement: the AdvisoryRelationship is minted with
`prov:wasGeneratedBy` pointing back to the HumanReview that created it. This
means you can always trace a coverage assignment back to the engagement event
that established it.

In [ ]:
# Step 5: Write Audit Trail to SLGD
print('Step 5: Audit Trail Written to SLGD')
print('=' * 60)

# Generate the audit trail triples
audit_triples = []
workflow_id = f'workflow-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
cust_uri = f'<{INST_NS}customer-{target_customer["customer_id"]}>'
signal_uri = f'<{INST_NS}signal-{workflow_id}>'
score_uri = f'<{INST_NS}score-{workflow_id}>'
route_uri = f'<{INST_NS}routing-{workflow_id}>'
review_uri = f'<{INST_NS}review-{workflow_id}>'
advisor_uri = f'<{INST_NS}advisor-alex-morgan>'
PROV_NS = 'http://www.w3.org/ns/prov#'

# Signal
audit_triples.append(f'{signal_uri} <{RDF_TYPE}> <{ATLAS_NS}WealthSignal> .')
audit_triples.append(f'{signal_uri} <{ATLAS_NS}hasSignalType> <{ATLAS_NS}LargeDepositPattern> .')
audit_triples.append(f'{cust_uri} <{ATLAS_NS}producesSignal> {signal_uri} .')

# Score (probabilistic-explainable)
audit_triples.append(f'{score_uri} <{RDF_TYPE}> <{ATLAS_NS}Score> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}scoreValue> "{score_value}"^^<{XSD_NS}decimal> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}probabilistic> "true"^^<{XSD_NS}boolean> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}explainability> "true"^^<{XSD_NS}boolean> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}modelVersion> "wealth-xgb-v1.0"^^<{XSD_NS}string> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}confidence> "{score_value}"^^<{XSD_NS}decimal> .')
audit_triples.append(f'{signal_uri} <{ATLAS_NS}hasScore> {score_uri} .')

# Routing decision (deterministic)
audit_triples.append(f'{route_uri} <{RDF_TYPE}> <{ATLAS_NS}RoutingDecision> .')
audit_triples.append(f'{route_uri} <{ATLAS_NS}selectedRoute> "{selected_route}"^^<{XSD_NS}string> .')
audit_triples.append(f'{signal_uri} <{ATLAS_NS}triggersRouting> {route_uri} .')

# Human review (engagement event)
audit_triples.append(f'{review_uri} <{RDF_TYPE}> <{ATLAS_NS}HumanReview> .')
audit_triples.append(f'{review_uri} <{RDF_TYPE}> <{PROV_NS}Activity> .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}reviewOutcome> "{review_outcome}"^^<{XSD_NS}string> .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}reviewDate> "{review_timestamp}"^^<{XSD_NS}dateTime> .')
audit_triples.append(f'{route_uri} <{ATLAS_NS}reviewedBy> {review_uri} .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}conductedBy> {advisor_uri} .')

# Advisor
audit_triples.append(f'{advisor_uri} <{RDF_TYPE}> <{ATLAS_NS}Advisor> .')
audit_triples.append(f'{advisor_uri} <http://www.w3.org/2000/01/rdf-schema#label> "Alex Morgan"^^<{XSD_NS}string> .')

# --- APPROVED branch: mint an AdvisoryRelationship (coverage assertion) ---
# When the review outcome is APPROVED, the workflow creates a standing coverage
# assignment. This is the second output of Module 8: not just an event record,
# but a persistent relationship that the CRM and future queries can rely on.
if review_outcome == 'APPROVED':
    advisory_rel_uri = f'<{INST_NS}advisory-rel-{workflow_id}>'
    rel_type_uri = f'<{ATLAS_NS}RelType_Primary>'

    # The AdvisoryRelationship instance
    audit_triples.append(f'{advisory_rel_uri} <{RDF_TYPE}> <{ATLAS_NS}AdvisoryRelationship> .')

    # coveringAdvisor: from HumanReview.conductedBy (Alex Morgan)
    audit_triples.append(f'{advisory_rel_uri} <{ATLAS_NS}coveringAdvisor> {advisor_uri} .')

    # advisesCustomer: the target customer from this workflow
    audit_triples.append(f'{advisory_rel_uri} <{ATLAS_NS}advisesCustomer> {cust_uri} .')

    # coverageStartDate: from the reviewDate (coverage begins at approval)
    review_date_only = review_timestamp.split('T')[0]
    audit_triples.append(f'{advisory_rel_uri} <{ATLAS_NS}coverageStartDate> "{review_date_only}"^^<{XSD_NS}date> .')

    # relationshipType: Primary (new wealth coverage assignment)
    audit_triples.append(f'{advisory_rel_uri} <{ATLAS_NS}relationshipType> {rel_type_uri} .')

    # prov:wasGeneratedBy: links coverage back to the engagement event
    audit_triples.append(f'{advisory_rel_uri} <{PROV_NS}wasGeneratedBy> {review_uri} .')

    # Link customer to the relationship
    audit_triples.append(f'{cust_uri} <{ATLAS_NS}hasAdvisor> {advisory_rel_uri} .')

    print(f'  APPROVED branch: AdvisoryRelationship minted')
    print(f'    coveringAdvisor:   Alex Morgan (from HumanReview.conductedBy)')
    print(f'    advisesCustomer:   {target_customer["first_name"]} {target_customer["last_name"]}')
    print(f'    coverageStartDate: {review_date_only} (from reviewDate)')
    print(f'    relationshipType:  RelType_Primary')
    print(f'    prov:wasGeneratedBy -> HumanReview (engagement -> coverage link)')
    print()

print(f'  Audit trail triples generated: {len(audit_triples)}')
print()
print(f'  The complete chain:')
print(f'    Customer -> producesSignal -> WealthSignal')
print(f'    WealthSignal -> hasScore -> Score (probabilistic-explainable)')
print(f'    RoutingDecision -> selectedRoute -> ROUTE_ADVISOR_QUEUE')
print(f'    RoutingDecision -> reviewedBy -> HumanReview')
print(f'    HumanReview -> conductedBy -> Advisor (Alex Morgan)')
print(f'    HumanReview -> reviewOutcome -> APPROVED')
print(f'    AdvisoryRelationship -> coveringAdvisor -> Advisor (coverage)')
print(f'    AdvisoryRelationship -> prov:wasGeneratedBy -> HumanReview (provenance)')
print()
print(f'  This is queryable via SPARQL (the CQ6 audit-trail query from Module 1).')

## The Audit Trail Query: One Query That Shows Everything

This query traverses the full chain produced by this workflow: signal detection, scoring, routing decision, human review, and coverage assignment — in a single SPARQL SELECT. Each variable in the result row corresponds to a distinct governance artifact.

**What it runs against here:** The query executes against `g_demo`, a local graph assembled from this walkthrough's `audit_triples`. This makes the result easy to read end-to-end with a single synthetic workflow.

**Why this is sound, not a shortcut:** The SPARQL query itself is identical to what runs in production — it is a pinned, version-controlled artifact, not generated per request (this is the same boundary established in Module 7, where the audit-trail query was deliberately excluded from LLM translation precisely because it belongs as a fixed artifact). In a live deployment, the same query points at the populated SLGD (Semantic Layer Graph Database) with all promoted customers; the query does not change — only the graph it addresses does.

In [ ]:
# Audit Trail Query — full chain from signal to coverage
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD as XSD_NS_RDF

ATLAS_RDF = Namespace('https://github.com/your-org/atlas/ontology#')
INST_RDF = Namespace('https://github.com/your-org/atlas/instance#')

# Build the audit trail as a local graph from this walkthrough's audit_triples
g_demo = Graph()
for triple_str in audit_triples:
    # Parse N-Triple into the graph
    parts = triple_str.rstrip(' .').split(' ', 2)
    if len(parts) == 3:
        s = URIRef(parts[0].strip('<>'))
        p = URIRef(parts[1].strip('<>'))
        if parts[2].startswith('<'):
            o = URIRef(parts[2].strip('<>'))
        elif '^^' in parts[2]:
            val, dtype = parts[2].rsplit('^^', 1)
            val = val.strip('"')
            dtype = URIRef(dtype.strip('<>'))
            o = Literal(val, datatype=dtype)
        else:
            o = Literal(parts[2].strip('"'))
        g_demo.add((s, p, o))

# Run the audit trail query against the local walkthrough graph
audit_trail_query = atlas_sparql.build_prefixes() + '''
SELECT ?customer ?signal ?signalType ?score ?scoreValue ?route ?reviewOutcome ?advisor WHERE {
    ?customer atlas:producesSignal ?signal .
    ?signal atlas:hasSignalType ?signalType ;
            atlas:hasScore ?score .
    ?score atlas:scoreValue ?scoreValue .
    ?signal atlas:triggersRouting ?routing .
    ?routing atlas:selectedRoute ?route ;
             atlas:reviewedBy ?review .
    ?review atlas:reviewOutcome ?reviewOutcome ;
            atlas:conductedBy ?advisor .
}
'''

print('Audit Trail Query')
print('=' * 60)
print()
print('Traversing: signal → score → routing → human review → advisory coverage')
print()

results = list(g_demo.query(atlas_sparql.validate(audit_trail_query)))
if results:
    for row in results:
        print(f'  Signal type:    {str(row.signalType).split("#")[-1]}')
        print(f'  Score:          {row.scoreValue}  (probabilistic; SHAP-attributed)')
        print(f'  Route selected: {row.route}  (from closed enumeration)')
        print(f'  Review outcome: {row.reviewOutcome}')
        print(f'  Advisor assigned: {str(row.advisor).split("#")[-1]}')
else:
    print('  (No results - check audit trail construction)')

print()
print('Each row is a complete, queryable audit record — signal to coverage in one query.')

In [ ]:
print('=' * 60)
print('MODULE 8 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Audit trail has all required components
required_types = ['WealthSignal', 'Score', 'RoutingDecision', 'HumanReview', 'Advisor', 'AdvisoryRelationship']
for rtype in required_types:
    found = any(rtype in t for t in audit_triples)
    status = 'PASS' if found else 'FAIL'
    print(f'[{status}] Gate 1.{required_types.index(rtype)+1} - {rtype} in audit trail')
    if not found: gate_pass = False

# Gate 2: Score has SHAP (explainability = true)
has_explainability = any('explainability' in t and 'true' in t for t in audit_triples)
print(f'[{"PASS" if has_explainability else "FAIL"}] Gate 2 - Score has explainability=true (SHAP)')
if not has_explainability: gate_pass = False

# Gate 3: Route is from closed set
has_valid_route = any('ROUTE_ADVISOR_QUEUE' in t or 'ROUTE_SUPPRESSION_LIST' in t or 'ROUTE_ESCALATION' in t for t in audit_triples)
print(f'[{"PASS" if has_valid_route else "FAIL"}] Gate 3 - Route from closed enumerated set')
if not has_valid_route: gate_pass = False

# Gate 4: Human review has outcome and advisor
has_outcome = any('reviewOutcome' in t for t in audit_triples)
has_advisor = any('conductedBy' in t for t in audit_triples)
print(f'[{"PASS" if has_outcome and has_advisor else "FAIL"}] Gate 4 - HumanReview has outcome + advisor')
if not (has_outcome and has_advisor): gate_pass = False

# Gate 4b: AdvisoryRelationship has required properties
has_covering = any('coveringAdvisor' in t for t in audit_triples)
has_advises = any('advisesCustomer' in t for t in audit_triples)
has_start = any('coverageStartDate' in t for t in audit_triples)
has_generated = any('wasGeneratedBy' in t for t in audit_triples)
all_ar = has_covering and has_advises and has_start and has_generated
print(f'[{"PASS" if all_ar else "FAIL"}] Gate 4b - AdvisoryRelationship has coveringAdvisor, advisesCustomer, coverageStartDate, prov:wasGeneratedBy')
if not all_ar: gate_pass = False

# Gate 5: Audit trail query returns results
print(f'[{"PASS" if results else "FAIL"}] Gate 5 - Audit trail query returns results')
if not results: gate_pass = False

print()
if gate_pass:
    print('MODULE 8 VALIDATION: PASS')
    print()
    print('Congratulations. You have completed the ATLAS workshop.')
    print('You can now:')
    print('  1. Articulate why the deterministic-vs-probabilistic boundary matters')
    print('  2. Build this pattern in your own account against your own data')
    print('  3. Defend the architecture in front of an MRM reviewer')
    print('  4. Extend the ontology with your institution\'s concepts')
else:
    print('MODULE 8 VALIDATION: FAIL')
    raise AssertionError('Module 8 validation gate failed.')

## What Changed

| Artifact | Location | Description |
|----------|----------|-------------|
| End-to-end workflow | This notebook | Simulated wealth-signal detection through advisor approval |
| Audit trail | Generated in cell 12 | Full PROV-O-attributed chain queryable via SPARQL |
| Audit trail query | Cell 14 | One query showing the complete chain from signal to coverage |

**You have a working reference implementation on this workshop's synthetic dataset.**

Every layer is operational:
- **Data Integration**: Three patterns feeding the LGD (Module 4)
- **Ontology and Digital Twin**: FIBO-aligned, SHACL-validated, two-tier Neptune (Modules 1-3, 5-6)
- **Application**: Bounded agent, human-in-the-loop, NL-to-SPARQL (Modules 7-8)

The deterministic-vs-probabilistic boundary is enforced by two specific mechanisms:
the SHACL shapes in Module 6 (run via `pyshacl` against the SLGD) and the
`atlas_sparql.validate()` pre-check in Module 7 (catches write operations from
the LLM before they reach Neptune). A reviewer can run the SHACL validator
against `atlas-shapes.ttl` and the audit trail is queryable end-to-end.

What's still needed to move from this reference implementation to production:
- Real source data via the R2RML mappings (Module 4 shows the patterns; production
  needs real Iceberg tables, real Athena workgroups, real Lambda integrations)
- A real scoring model with model-card documentation (Module 8 uses a placeholder)
- PII redaction at the Bedrock layer (Bedrock Guardrails, configured but not
  exercised against real PII in this workshop)
- Operational runbooks for the human-in-the-loop step (the workshop simulates
  Alex Morgan; production needs real reviewers with trained workflows)

## Next Steps

- **Run the cleanup notebook** (`99-cleanup`) to tear down all infrastructure
- **Read `docs/follow-on-labs.md`** for the real-time depth lab and external-signals lab
- **Replace the synthetic data** with your institution's data using the Extending
  This to Your Data appendices from each module
- **Walk through the end-to-end audit trail** using the query from cell 14